# L3 — Orchestrator Harness
## What You'll Learn
- How to create an AgentCore Harness that orchestrates specialist agents discovered via the Registry
- How to configure AgentCore Memory with short-term and long-term extraction strategies
- How to send the Harness's traces to **Datadog** via OTLP-direct — environment variables only, no Agent, Collector, or custom container
- How one OTLP exporter feeds **both Datadog APM and Agent Observability**, and what the `dd-otlp-source` header changes
- How observability is **split by signal**: traces → Datadog, while logs & metrics stay in CloudWatch
- How to invoke the Harness end-to-end and stream the response
## Overview
The Harness is the top-level orchestrator for AnyCompany's customer support system. It holds the conversation with the customer, discovers specialist agents (Order Agent, Refund Agent) through the AgentCore Registry, and delegates tasks to them. It uses Claude Sonnet 4.6 on Bedrock directly — not through LiteLLM — and maintains conversation state via AgentCore Memory (short-term chat history and long-term extracted knowledge).

For observability, the Harness sends its **traces** to Datadog through OpenTelemetry (OTLP-direct, configured with environment variables in Step 5), while **logs and metrics** continue to flow to CloudWatch (AgentCore Observability, automatic). Only the trace signal is redirected; nothing else changes.

Those traces land in **two Datadog products at once**. They arrive in APM as distributed spans, and because they carry OpenTelemetry's GenAI semantic conventions, Datadog also converts them into **Agent Observability** spans - model, token counts, cost, and request parameters as structured fields. One exporter, two views of the same call, cross-linked in the UI.
## Architecture
```
  Customer
       │
       ▼
  ┌─────────────────────────────────────────────────────────────────┐
  │  AgentCore Harness (anycompany_orchestrator)                    │
  │                                                                 │
  │  Model: Claude Sonnet 4.6 on Bedrock (direct, no LiteLLM)       │
  │  Tools: registry-tools (via harness gateway) + Code Interpreter │
  │  Memory: AgentCore Memory (short-term + long-term)              │
  └─────────────────────────────────────────────────────────────────┘
                │  registry-tools___search_agents / invoke_agent
                ▼
   anycompany-harness-gateway  (MCP, AWS_IAM)
        └─ anycompany_registry_tools Lambda
              ├─ search  → discovers agents in the Registry
              └─ invoke  → A2A call to a specialist runtime
                │
       ┌────────┴─────────┐
       ▼                  ▼
  Order Agent        Refund Agent
  (A2A, Runtime)     (A2A, Runtime)
       └────────┬─────────┘
                ▼
   anycompany-agent-gateway  (MCP, Cognito M2M client_credentials)
        └─ order / refund Lambda tools  → DynamoDB / KB

  Observability (split by signal):
    Traces           ──(OTLP http/protobuf, via env vars)──►  Datadog APM
                                                          └─►  Agent Observability
                                                            (service / ml_app: anycompany-orchestrator)
    Logs & metrics   ──(AgentCore Observability, automatic)──►  CloudWatch
```
## Steps
1. Load configuration from SSM
2. Create the IAM execution role for the Harness
3. Create AgentCore Memory
4. Enable observability — log delivery for Memory and Gateway (logs & metrics stay in CloudWatch)
5. Create the AgentCore Harness (with Datadog OTLP tracing enabled)
6. Test the Harness with a sample invocation
7. Save resource IDs to SSM


## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites
- `1_setup_resources.ipynb` completed — Registry, Gateway, and Cognito IDs **and Datadog credentials** (`dd_api_key`, `dd_site`) must be in SSM (stored in that notebook's Step 6)
- `1_pluggable_inference_layer.ipynb` (L2) completed — `harness_model_id` must be in SSM
- `2_order_agent.ipynb` and `3_refund_agent.ipynb` completed — agents must be registered in the Registry
- `boto3`, `bedrock-agentcore`, and `strands-agents` (installed in the first code cell)


Install required packages.

In [1]:
%pip install --quiet --upgrade boto3 botocore>=1.43.58 bedrock-agentcore==1.14.0 strands-agents==1.43.0


Note: you may need to restart the kernel to use updated packages.


Import libraries, set the region, and initialise AWS clients.

In [2]:
import boto3
import json
import os
import uuid
import time

# --- Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region

# Clients
from botocore.config import Config
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION,
                                  config=Config(read_timeout=300))
sts = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

Account: 359894365202
Region:  us-west-2


## Step 1: Load Configuration from SSM

Fetch Registry and Gateway IDs from SSM — created by `1_pre-requisites.ipynb`.

In [3]:
SSM_PREFIX = "/anycompany/agentcore"
ssm = boto3.client("ssm", region_name=REGION)

def get_ssm(name: str) -> str:
    """Retrieve a single SSM parameter value."""
    return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]

REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
REGISTRY_ARN = get_ssm(f"{SSM_PREFIX}/registry_arn")
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")

print(f"Registry ID:  {REGISTRY_ID}")
print(f"Registry ARN: {REGISTRY_ARN}")
print(f"Gateway ID:   {GATEWAY_ID}")
print(f"Gateway ARN:  {GATEWAY_ARN}")

Registry ID:  XCtvGlP9E1u5ZoYr
Registry ARN: arn:aws:agent-registry:us-west-2:359894365202:registry/XCtvGlP9E1u5ZoYr
Gateway ID:   anycompany-agent-gateway-m1jnn7lvju
Gateway ARN:  arn:aws:bedrock-agentcore:us-west-2:359894365202:gateway/anycompany-agent-gateway-m1jnn7lvju


## Step 2: Create the IAM Execution Role

Create the IAM role the Harness assumes at runtime, with permissions for Bedrock, AgentCore Memory, Registry, Gateway, Code Interpreter, and CloudWatch.

In [4]:
iam = boto3.client("iam")
ROLE_NAME = "AnyCompanyAgentCoreHarnessRole"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "bedrock-agentcore.amazonaws.com",
                    "bedrock.amazonaws.com"
                ]
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

# CloudWatch metrics require Resource:* but namespace condition limits scope to bedrock-agentcore
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelInvocation",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*"
            ]
        },
        {
            "Sid": "EcrPublicTokenAccess",
            "Effect": "Allow",
            "Action": [
                "ecr-public:GetAuthorizationToken",
                "sts:GetServiceBearerToken"
            ],
            "Resource": "*"
        },
        {
            "Sid": "CloudWatchMetricsPublish",
            "Effect": "Allow",
            "Action": "cloudwatch:PutMetricData",
            # PutMetricData requires Resource: '*' but is scoped to the
            # bedrock-agentcore namespace via the Condition below.
            "Resource": "*",
            "Condition": {
                "StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}
            }
        },
        {
            "Sid": "CloudWatchLogsGroup",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:DescribeLogStreams"
            ],
            # Wildcard required for dynamic runtime log groups under /aws/bedrock-agentcore/runtimes/.
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtimes/*"
        },
        {
            "Sid": "CloudWatchLogStream",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogStream",
                "logs:PutLogEvents",
                "logs:GetLogEvents"
            ],
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtimes/*:log-stream:*"
        },
        {
            "Sid": "CloudWatchLogsDescribeGroups",
            "Effect": "Allow",
            "Action": "logs:DescribeLogGroups",
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:*"
        },
        {
            "Sid": "AgentCoreWorkloadIdentity",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetWorkloadAccessToken",
                "bedrock-agentcore:GetWorkloadAccessTokenForJWT"
            ],
            # Wildcard required for harness workload identities — scoped to harness_* prefix.
            "Resource": [
                f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default",
                f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default/workload-identity/harness_*"
            ]
        },
        {
            "Sid": "AgentCoreBrowserDefault",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:StartBrowserSession",
                "bedrock-agentcore:StopBrowserSession",
                "bedrock-agentcore:GetBrowserSession",
                "bedrock-agentcore:ListBrowserSessions",
                "bedrock-agentcore:UpdateBrowserStream",
                "bedrock-agentcore:ConnectBrowserAutomationStream",
                "bedrock-agentcore:ConnectBrowserLiveViewStream"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:browser/*"
        },
        {
            "Sid": "AgentCoreCodeInterpreterDefault",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:StartCodeInterpreterSession",
                "bedrock-agentcore:StopCodeInterpreterSession",
                "bedrock-agentcore:GetCodeInterpreterSession",
                "bedrock-agentcore:ListCodeInterpreterSessions",
                "bedrock-agentcore:InvokeCodeInterpreter"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:code-interpreter/*"
        },
        {
            "Sid": "AgentCoreMemory",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateEvent",
                "bedrock-agentcore:DeleteEvent",
                "bedrock-agentcore:GetEvent",
                "bedrock-agentcore:ListEvents",
                "bedrock-agentcore:RetrieveMemoryRecords"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:memory/*"
        },
        {
        "Sid": "AgentCoreInvokeRuntime",
        "Effect": "Allow",
        "Action": "bedrock-agentcore:InvokeAgentRuntime",
        "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:runtime/*"
        },
        {
            "Sid": "AgentCoreRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:SearchRegistryRecords",
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:ListRegistryRecords"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:*"
        },
        {
            "Sid": "AgentCoreGatewayAccess",
            "Effect": "Allow",
            "Action": "bedrock-agentcore:InvokeGateway",
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/*"
        },
        {
            "Sid": "AgentCoreAgentDiscovery",
            "Effect": "Allow",
            # The Harness finds a specialist agent in the Registry, then resolves its
            # runtime + endpoint and fetches its A2A agent card before delegating.
            # Without these, GetAgentCard / ListAgentRuntimes return 403, the Harness
            # can't delegate, and the loop runs until maxIterations ("Max iterations
            # exceeded"). InvokeAgentRuntime (above) then actually calls the agent.
            "Action": [
                "bedrock-agentcore:ListAgentRuntimes",
                "bedrock-agentcore:GetAgentRuntime",
                "bedrock-agentcore:GetAgentRuntimeEndpoint",
                "bedrock-agentcore:ListAgentRuntimeEndpoints",
                "bedrock-agentcore:GetAgentCard"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:*"
        }
    ]
}

try:
    role_response = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AnyCompany AgentCore Harness"
    )
    EXECUTION_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created role: {EXECUTION_ROLE_ARN}")
    _created = True
except iam.exceptions.EntityAlreadyExistsException:
    EXECUTION_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    print(f"Role already exists: {EXECUTION_ROLE_ARN}")
    _created = False

# Always (re)attach the inline policy so re-running this cell applies permission
# changes to an EXISTING role (e.g. the agent-discovery actions added above).
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="AnyCompanyHarnessPermissions",
    PolicyDocument=json.dumps(permissions_policy)
)
print("Attached/updated inline policy.")
if _created:
    print("Waiting 10s for IAM propagation...")
    time.sleep(10)

print(f"\nExecution Role ARN: {EXECUTION_ROLE_ARN}")

Created role: arn:aws:iam::359894365202:role/AnyCompanyAgentCoreHarnessRole
Attached/updated inline policy.
Waiting 10s for IAM propagation...

Execution Role ARN: arn:aws:iam::359894365202:role/AnyCompanyAgentCoreHarnessRole


## Step 3: Create AgentCore Memory

Short-term memory (raw conversation events) is automatic. Long-term memory requires explicit strategy configuration — we define three: session summarization, user preference extraction, and semantic fact extraction.

Create the memory resource with three long-term extraction strategies scoped per `actorId`.

In [5]:
MEMORY_NAME = "anycompany_agent_memory"
print(f"Creating memory: {MEMORY_NAME}")

try:
    memory_response = agentcore_control.create_memory(
        name=MEMORY_NAME,
        description="long-term memory for the AnyCompany orchestrator harness.",
        memoryExecutionRoleArn=EXECUTION_ROLE_ARN,
        eventExpiryDuration=30,  # Events expire after 1 hour
        memoryStrategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "session_summary",
                    "description": "Summarizes each conversation session",
                    # `namespaceTemplates` (not the legacy `namespaces`) supports
                    # variable substitution. {actorId}/{sessionId} ensures each
                    # user's session summaries are isolated from other users'.
                    "namespaceTemplates": ["summaries/{actorId}/{sessionId}"]
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "user_preferences",
                    "description": "Extracts stated user preferences",
                    # Per-user preference scoping. Without {actorId} every
                    # user's preferences would collide in a flat namespace.
                    "namespaceTemplates": ["preferences/{actorId}"]
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "semantic_facts",
                    "description": "Extracts factual knowledge for semantic retrieval",
                    "namespaceTemplates": ["facts/{actorId}"]
                }
            }
        ]
    )

    MEMORY_ID = memory_response["memory"]["id"]
    MEMORY_ARN = memory_response["memory"]["arn"]
    print(f"Created memory: {MEMORY_NAME}")
    print(f"  ID:  {MEMORY_ID}")
    print(f"  ARN: {MEMORY_ARN}")

except (agentcore_control.exceptions.ConflictException, agentcore_control.exceptions.ValidationException):
    print(f"Memory '{MEMORY_NAME}' already exists. Looking up...")
    memories = agentcore_control.list_memories()
    for mem in memories.get("memories", []):
        # list_memories doesn't return name, so get each one
        detail = agentcore_control.get_memory(memoryId=mem["id"])
        if detail["memory"].get("name") == MEMORY_NAME:
            MEMORY_ID = detail["memory"]["id"]
            MEMORY_ARN = detail["memory"]["arn"]
            print(f"  ID:  {MEMORY_ID}")
            print(f"  ARN: {MEMORY_ARN}")
            break
    else:
        raise RuntimeError(f"Could not find memory with name '{MEMORY_NAME}'")

Creating memory: anycompany_agent_memory
Created memory: anycompany_agent_memory
  ID:  anycompany_agent_memory-qDC7LT5khH
  ARN: arn:aws:bedrock-agentcore:us-west-2:359894365202:memory/anycompany_agent_memory-qDC7LT5khH


Poll until the Memory resource reaches ACTIVE status.

In [6]:
# Wait for memory to be ready
import time as _time

print("Waiting for memory to be ready...")
_mem_start = _time.time()
_poll = 0

while True:
    m = agentcore_control.get_memory(memoryId=MEMORY_ID)
    status = m["memory"]["status"]
    _elapsed = _time.time() - _mem_start
    _poll += 1
    if status == "ACTIVE":
        print(f"Memory is ACTIVE after {_elapsed:.1f}s ({_poll} poll(s))")
        break
    elif status == "FAILED":
        reason = m["memory"].get("failureReason", "unknown")
        raise RuntimeError(f"Memory creation failed: {reason}")
    print(f"  [{_elapsed:5.1f}s] Status: {status} — waiting... (poll #{_poll})")
    time.sleep(5)

print(f"\nMemory ID:  {MEMORY_ID}")
print(f"Memory ARN: {MEMORY_ARN}")

Waiting for memory to be ready...
  [  0.2s] Status: CREATING — waiting... (poll #1)
  [  5.3s] Status: CREATING — waiting... (poll #2)
  [ 10.5s] Status: CREATING — waiting... (poll #3)
  [ 15.6s] Status: CREATING — waiting... (poll #4)
  [ 20.8s] Status: CREATING — waiting... (poll #5)
  [ 26.0s] Status: CREATING — waiting... (poll #6)
  [ 31.2s] Status: CREATING — waiting... (poll #7)
  [ 36.3s] Status: CREATING — waiting... (poll #8)
  [ 41.5s] Status: CREATING — waiting... (poll #9)
  [ 46.6s] Status: CREATING — waiting... (poll #10)
  [ 51.8s] Status: CREATING — waiting... (poll #11)
  [ 56.9s] Status: CREATING — waiting... (poll #12)
  [ 62.1s] Status: CREATING — waiting... (poll #13)
  [ 67.2s] Status: CREATING — waiting... (poll #14)
  [ 72.4s] Status: CREATING — waiting... (poll #15)
  [ 77.5s] Status: CREATING — waiting... (poll #16)
  [ 82.7s] Status: CREATING — waiting... (poll #17)
  [ 87.8s] Status: CREATING — waiting... (poll #18)
  [ 93.0s] Status: CREATING — waiting..

## Step 4: Enable Observability — logs & metrics in CloudWatch

Telemetry in this workshop is split **by signal**:

- **Traces → Datadog.** The Harness exports its traces to Datadog's OTLP traces intake
  (configured in Step 5, below), where they populate both APM and Agent Observability.
  The specialist agents send their traces via the ddtrace sidecar instead — a different
  instrumentation stack, chosen because we control that container image.
- **Logs & metrics → CloudWatch.** The Harness, Memory, and Gateway are **AWS-managed**
  and emit logs and metrics to CloudWatch. The **Harness does this automatically**
  (AgentCore Observability — no setup, from the first invocation). This step only wires
  up **log delivery** for **Memory** and **Gateway** so their `APPLICATION_LOGS` are
  captured too.

Only the *trace* signal is redirected to Datadog; logs and metrics are untouched and
keep flowing to CloudWatch. Datadog's AWS Integration can optionally forward those
CloudWatch logs/metrics so everything sits alongside the traces.


Configure CloudWatch log delivery for the Memory resource.

In [7]:
logs_client = boto3.client("logs", region_name=REGION)

MEMORY_LOG_PREFIX = "/aws/vendedlogs/bedrock-agentcore/memory"

for log_type in ["APPLICATION_LOGS"]:
    source_name = f"mem-{MEMORY_ID[:20]}-{log_type.lower()}"
    dest_log_group = f"{MEMORY_LOG_PREFIX}/{log_type}/{MEMORY_ID}"
    dest_name = f"mem-{MEMORY_ID[:20]}-{log_type.lower()}-dest"

    # Check if source already exists
    existing_source = None
    sources = logs_client.describe_delivery_sources().get('deliverySources', [])
    for s in sources:
        if MEMORY_ARN in s.get('resourceArns', []) and s.get('logType') == log_type:
            existing_source = s['name']
            break

    if existing_source:
        source_name = existing_source
        print(f"✓ {log_type} source already exists: {source_name}")
    else:
        try:
            logs_client.put_delivery_source(name=source_name, resourceArn=MEMORY_ARN, logType=log_type)
            print(f"✓ Created {log_type} source: {source_name}")
        except Exception as e:
            print(f"✗ {log_type} source: {e}")

    # Create log group
    try:
        logs_client.create_log_group(logGroupName=dest_log_group)
    except logs_client.exceptions.ResourceAlreadyExistsException:
        pass

    # Create destination
    try:
        logs_client.put_delivery_destination(
            name=dest_name,
            deliveryDestinationConfiguration={
                "destinationResourceArn": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{dest_log_group}"
            },
        )
    except logs_client.exceptions.ConflictException:
        pass

    # Create delivery
    try:
        logs_client.create_delivery(
            deliverySourceName=source_name,
            deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{dest_name}",
        )
        print(f"✓ {log_type} delivery created")
    except logs_client.exceptions.ConflictException:
        print(f"✓ {log_type} delivery already exists")
    except Exception as e:
        print(f"✗ {log_type} delivery: {e}")

print(f"\nMemory observability configured for {MEMORY_ID}")

✓ Created APPLICATION_LOGS source: mem-anycompany_agent_mem-application_logs
✓ APPLICATION_LOGS delivery created

Memory observability configured for anycompany_agent_memory-qDC7LT5khH


Configure CloudWatch log delivery for the Gateway resource.

In [8]:
# Fetch Gateway config from SSM
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
print(f"Gateway: {GATEWAY_ID}")

GATEWAY_LOG_PREFIX = "/aws/vendedlogs/bedrock-agentcore/gateway"

for log_type in ["APPLICATION_LOGS"]:
    dest_log_group = f"{GATEWAY_LOG_PREFIX}/{log_type}/{GATEWAY_ID}"
    source_name = f"gw-{GATEWAY_ID[:20]}-{log_type.lower()}"
    dest_name = f"gw-{GATEWAY_ID[:20]}-{log_type.lower()}-dest"

    print(f"Configuring {log_type}...")

    # Check if a delivery source already exists for this gateway + log type
    existing_source = None
    sources = logs_client.describe_delivery_sources().get('deliverySources', [])
    for s in sources:
        if GATEWAY_ARN in s.get('resourceArns', []) and s.get('logType') == log_type:
            existing_source = s['name']
            break

    if existing_source:
        source_name = existing_source
        print(f"  ✓ Using existing delivery source: {source_name}")
    else:
        try:
            logs_client.put_delivery_source(name=source_name, resourceArn=GATEWAY_ARN, logType=log_type)
            print(f"  ✓ Created delivery source: {source_name}")
        except Exception as e:
            print(f"  ✗ Delivery source error: {e}")

    # Create log group
    try:
        logs_client.create_log_group(logGroupName=dest_log_group)
        print(f"  ✓ Created log group: {dest_log_group}")
    except logs_client.exceptions.ResourceAlreadyExistsException:
        print(f"  ✓ Log group already exists: {dest_log_group}")

    # Create delivery destination
    try:
        logs_client.put_delivery_destination(
            name=dest_name,
            deliveryDestinationConfiguration={
                "destinationResourceArn": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{dest_log_group}"
            },
        )
        print(f"  ✓ Created delivery destination: {dest_name}")
    except logs_client.exceptions.ConflictException:
        print(f"  ✓ Delivery destination already exists: {dest_name}")

    # Create delivery
    try:
        logs_client.create_delivery(
            deliverySourceName=source_name,
            deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{dest_name}",
        )
        print(f"  ✓ Created delivery")
    except logs_client.exceptions.ConflictException:
        print(f"  ✓ Delivery already exists")
    except Exception as e:
        print(f"  ✗ Delivery error: {e}")

print(f"\nGateway log delivery configured.")


Gateway: anycompany-agent-gateway-m1jnn7lvju
Configuring APPLICATION_LOGS...
  ✓ Created delivery source: gw-anycompany-agent-gat-application_logs
  ✓ Created log group: /aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/anycompany-agent-gateway-m1jnn7lvju
  ✓ Created delivery destination: gw-anycompany-agent-gat-application_logs-dest
  ✓ Created delivery

Gateway log delivery configured.


## Step 5: Create the AgentCore Harness

Define the system prompt that governs the Harness's orchestration behaviour.

In [9]:
SYSTEM_PROMPT = f"""You are the AnyCompany Electronics customer support orchestrator. Your job is to help
customers with order inquiries, refund requests, and product damage claims.

## Distinguishing statements from requests

The customer may STATE preferences or facts ('I prefer email for refund
updates') or REQUEST actions ('what is the status of ORD-10001?'). These
look similar but require different handling:

- A STATEMENT of preference, fact, or contact detail: acknowledge
  naturally in ONE sentence and offer to help with anything else. Do
  NOT trigger order lookups, refund flows, or specialist-agent calls.
  Memory extraction records it automatically; you do not need to act.

- A REQUEST for action ('check my order', 'process a refund', 'what
  is the status of', 'can you'): run the relevant workflow.

If a single message contains both ('My preferred contact is email and
I want to know the status of ORD-10001'), briefly acknowledge the
preference, then handle the request.

The presence of words like 'refund', 'order', or 'product' does NOT
automatically mean the customer wants a workflow. Read the verb:

'I prefer', 'I like', 'I am', 'my X is'      → STATEMENT (acknowledge only)
'can you', 'I want', 'I need', 'check', 'what is'  → REQUEST (run workflow)

## Agent Discovery

## YOUR TOOLS (use EXACTLY these names)

1. registry-tools___search_agents
   - Purpose: Find specialist agents in the registry
   - Parameters: {{"action": "search", "search_query": "<what you need>"}}
   - Returns: List of agents with name, description, agentRuntimeArn

2. registry-tools___invoke_agent
   - Purpose: Send a message to a specialist agent
   - Parameters: {{"action": "invoke", "agent_runtime_arn": "<ARN from search>", "message": "<your request>"}}
   - Returns: The agent's response text

3. code_interpreter
   - Purpose: Run Python code for date calculations
   - Use for: Getting today's date, computing days between dates, calculating refund amounts

## How You Work

1. When a customer asks for help, use the search_agents tool to find relevant agents
2. Use the discovered agent information to delegate tasks via the invoke_agent tool
3. For calculations (prorated amounts, partial refunds), use the code_interpreter tool
4. Summarize results clearly and confirm next steps with the customer

## Refund Eligibility Calculation Rules

When a customer requests a refund, follow these steps:

### Step 1: Get order and refund policy details
- Delegate to the Order Agent to retrieve order details
- Delegate to the Refund Agent to retrieve the refund policy for the customer's tier and order.
- The Refund Agent will return the return window days, refund percentage, and eligibility based on the Knowledge Base.

### Step 2: Calculation for refund
- When calculating refund eligibility,
  1. Use the code_interpreter tool to get the current date: from datetime import date; print(date.today())
  2. Parse the delivery date from the order details and the return window days from the Refund Agent's response.
  3. Calculates days_since_delivery = Find the number of days between current date and date of delivery.
  4. Apply the tier-based rules returned by the Refund Agent — do NOT use hardcoded rules.

### Step 3: Report the result
- State the tier, delivery date, days since delivery, eligibility, and refund amount
- If eligible, mention that they are eligible for return.
- If not eligible, explain why (return window expired)

IMPORTANT: Always use code_interpreter for date math — do NOT calculate in your head.
IMPORTANT: Always get refund policy rules from the Refund Agent — do NOT use hardcoded tier rules.

## Memory

You have access to long-term memory scoped to the current customer.
AgentCore automatically retrieves relevant past context (preferences,
extracted facts, prior session summaries) and injects it before each
turn. The injected context may be EMPTY — that is a real and common
signal, not an error.

Behave accordingly:

- When the customer asks about a preference or fact: answer ONLY from
  what is actually present in the retrieved memory above. If retrieval
  is empty, or the specific detail is not there, say so honestly and
  ask the customer to share. Do NOT confabulate or guess.

- When the customer states a preference, fact, or contact detail,
  acknowledge naturally in one sentence. Extraction is automatic;
  you do not have or need a 'store this' tool.

- Quote or paraphrase what is actually in retrieved memory rather
  than inventing details. If memory says 'Prefers email', you may
  reply 'you mentioned previously you prefer email'. Do not invent
  specifics that are not present in the retrieved text.

- Recently extracted memories may take a few minutes to surface
  across sessions; within a single session, recent messages are
  always visible.

CRITICAL: Never invent memories. If the retrieved context is empty,
the correct answer is 'I don't have that on record yet — could you
share it?' Saying you remember something you do not is a serious
failure.

## Response Style

Since this is a workshop demo, show your reasoning so attendees can
follow the orchestration flow:

- Briefly state what you are doing at each step (e.g., "Searching the
  registry for an order specialist...", "Delegating to the Refund Agent
  to check eligibility...")
- When you delegate to a specialist agent, mention which agent and why
- If a calculation is involved, show the inputs and the math
- Keep it concise — one line per step, not a wall of text
- After showing your work, deliver the final answer clearly to the
  customer, separated by a line break

Do NOT expose raw JSON payloads, full stack traces, or internal error
messages. If a tool call fails, briefly note the issue and retry or
explain what went wrong in plain language.

## Important Rules

- Order, refund, and account details may come ONLY from a SUCCESSFUL
  specialist tool response. Never produce them from memory, prior turns,
  the session context, or your own assumptions.
- If a specialist tool call fails, errors, times out, is denied or blocked,
  or the agent reports it is unavailable: tell the customer you could NOT
  retrieve the information and cannot complete that request right now. Do NOT
  fabricate, estimate, infer, or recall order/refund/account details in that
  case. The authenticated customer_id identifies WHO you are serving — it is
  NOT a source of order data.
- Never approve a refund without checking eligibility
- Only display orders, refund details, or account information belonging to
  the authenticated customer (the customer_id in the current session).
  If a query references a different customer's ID, politely decline and
  explain you can only assist with their own account.
- When delegating to the Order Agent, ALWAYS include the authenticated
  customer_id in your message so the lookup is scoped to that customer
  (order lookups now require customer_id).
- If an order lookup returns 'not found' or 'does not exist for your
  account', simply tell the customer 'That order does not belong to your
  account.' Do NOT list their other orders or suggest alternatives.
- Use search_agents for discovering agents, invoke_agent for calling them
- Use code_interpreter for math and data analysis
- Be empathetic and professional in all interactions
"""

print("System prompt defined.")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

System prompt defined.
Length: 7301 characters


Create the Harness with Claude Sonnet 4.6, the system prompt, Memory, Code Interpreter, and the registry-tools Gateway (agent discovery/invocation). Updates the existing Harness if it already exists.

**Datadog tracing (OTLP-direct).** The Harness runs AWS's OpenTelemetry (ADOT) distro,
so end-to-end tracing to Datadog is enabled purely through **environment variables** —
no Datadog Agent, no OpenTelemetry Collector, and no custom container. Its traces
(agent loop → model calls → tool calls) are sent to Datadog's **OTLP traces intake**
endpoint (`https://trace.agent.<site>/v1/traces`). This repoints only the *trace*
signal; logs and metrics still flow to CloudWatch (Step 4).

Three of the settings below decide **which Datadog products** receive these spans:

| Setting | Effect |
|---|---|
| `compute_stats=true` | Datadog computes APM trace metrics for the spans |
| `dd-otlp-source=llmobs` | Runs the GenAI conversion so the spans also populate **Agent Observability** |
| `OTEL_SEMCONV_STABILITY_OPT_IN` | Opts into the GenAI semantic conventions 1.37+ that the conversion requires |

Both headers work on the same exporter, so you get the APM view (the span in full
distributed context) and the Agent Observability view (model, tokens, cost, parameters) of
the same call, with a link between them in the UI.

In Agent Observability the Harness appears under the ML application
**`anycompany-orchestrator`**, derived from `OTEL_SERVICE_NAME`. The specialist agents set
`DD_LLMOBS_ML_APP=anycompany-customer-support` instead, so a delegated request spans **two ML
applications** - use the ML app selector to switch between them. The two instrumentation
paths are compared in `l6-observability/3_datadog_agent_observability.ipynb`.

Allow 3-5 minutes for these spans to appear in Agent Observability; APM shows them within
seconds.

> **Note:** Datadog's direct OTLP traces intake is in **Preview** and access-gated — the
> Datadog org must be allowlisted or the intake returns `403` (contact your Datadog CSM).
> The Harness also needs outbound egress to the Datadog endpoint (default `PUBLIC`
> network mode provides this).


In [10]:
print("Creating AgentCore Harness...")
HARNESS_NAME = "anycompany_orchestrator"
DD_API_KEY = get_ssm(f"{SSM_PREFIX}/dd_api_key")
DD_SITE    = get_ssm(f"{SSM_PREFIX}/dd_site")


try:
    harness_response = agentcore_control.create_harness(
        harnessName=HARNESS_NAME,
        executionRoleArn=EXECUTION_ROLE_ARN,

        # Model
        model={
            "bedrockModelConfig": {
                # AgentCore's create_harness API is AWS-native and only
                # supports Bedrock. It expects a bare Bedrock model ID.
                # L2 publishes this separately from the worker `model_id`
                # so the orchestrator stays on Bedrock even when workers
                # are swapped to a non-Bedrock provider via LiteLLM.
                "modelId": get_ssm(f"{SSM_PREFIX}/harness_model_id"),
                "maxTokens": 4096,
                "temperature": 0.1
            }
        },

        # System prompt
        systemPrompt=[{"text": SYSTEM_PROMPT}],

        # Tools — gateway (for registry discovery) + code interpreter
        tools=[
            {"type": "agentcore_gateway", "name": "registry-tools", "config": {"agentCoreGateway": {"gatewayArn": get_ssm(f"{SSM_PREFIX}/harness_gateway_arn")}}},
            {"type": "agentcore_code_interpreter", "name": "code_interpreter", "config": {"agentCoreCodeInterpreter": {}}},
        ],

        allowedTools=["*"],

        # Memory
        memory={
            "agentCoreMemoryConfiguration": {
                "arn": MEMORY_ARN,
                "messagesCount": 20
            }
        },

        # Execution limits
        maxIterations=20,
        maxTokens=16384,
        timeoutSeconds=300,
        # ── Datadog tracing via OTLP-direct (traces only; logs & metrics stay in CloudWatch) ──
        # The Harness runs AWS's OpenTelemetry (ADOT) distro, so tracing to Datadog needs
        # environment variables only — no Datadog Agent, no Collector, no custom container.
        #
        #   * Protocol MUST be http/protobuf; the Datadog OTLP intake rejects gRPC.
        #   * service / env come from OTEL_SERVICE_NAME and OTEL_RESOURCE_ATTRIBUTES, not
        #     ddtrace's DD_SERVICE / DD_ENV — the OTel exporter ignores those.
        #   * compute_stats=true    -> Datadog computes APM trace metrics for these spans.
        #   * dd-otlp-source=llmobs -> the same spans also populate Agent Observability.
        #   * environmentVariables is a top-level harness param; environmentArtifact is only
        #     for a custom container image.
        #
        # The OTLP traces intake is in Preview and access-gated: the org must be allowlisted
        # or the intake returns 403 (contact your Datadog CSM).
        environmentVariables={
            "OTEL_EXPORTER_OTLP_TRACES_PROTOCOL": "http/protobuf",   # Datadog OTLP intake rejects gRPC
            "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT": f"https://trace.agent.{DD_SITE}/v1/traces",
            "OTEL_EXPORTER_OTLP_TRACES_HEADERS": f"dd-api-key={DD_API_KEY},compute_stats=true,dd-otlp-source=llmobs",
            "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",  # GenAI semconv 1.37+
            "OTEL_SERVICE_NAME": "anycompany-orchestrator",          # -> Datadog `service` + `ml_app`
            "OTEL_RESOURCE_ATTRIBUTES": "deployment.environment=demo,deployment.environment.name=demo",  # -> Datadog `env`
        },

        tags={
            "project": "anycompany-workshop",
            "layer": "l3-orchestration",
            "role": "orchestrator"
        }
    )

    harness = harness_response["harness"]
    HARNESS_ID = harness["harnessId"]
    HARNESS_ARN = harness["arn"]
    print(f"\nHarness created!")
    print(f"  Name:   {harness['harnessName']}")
    print(f"  ID:     {HARNESS_ID}")
    print(f"  ARN:    {HARNESS_ARN}")
    print(f"  Status: {harness['status']}")

except agentcore_control.exceptions.ConflictException:
    print(f"Harness '{HARNESS_NAME}' already exists. Updating with new system prompt...")
    harnesses = agentcore_control.list_harnesses()
    for h in harnesses.get("harnesses", []):
        if h.get("harnessName") == HARNESS_NAME:
            HARNESS_ID = h["harnessId"]
            HARNESS_ARN = h["arn"]
            break
    else:
        raise RuntimeError(f"Could not find harness with name '{HARNESS_NAME}'")

    agentcore_control.update_harness(
        harnessId=HARNESS_ID,
        executionRoleArn=EXECUTION_ROLE_ARN,
        model={
            "bedrockModelConfig": {
                "modelId": get_ssm(f"{SSM_PREFIX}/harness_model_id"),
                "maxTokens": 4096,
                "temperature": 0.1
            }
        },
        systemPrompt=[{"text": SYSTEM_PROMPT}],
        tools=[
            {"type": "agentcore_gateway", "name": "registry-tools", "config": {"agentCoreGateway": {"gatewayArn": get_ssm(f"{SSM_PREFIX}/harness_gateway_arn")}}},
            {"type": "agentcore_code_interpreter", "name": "code_interpreter", "config": {"agentCoreCodeInterpreter": {}}}
        ],
        allowedTools=["*"],
        memory={"optionalValue": {"agentCoreMemoryConfiguration": {"arn": MEMORY_ARN, "messagesCount": 20}}},
        maxIterations=20,
        maxTokens=16384,
        timeoutSeconds=300,
        # ── Datadog tracing via OTLP-direct (traces only; logs & metrics stay in CloudWatch) ──
        # The Harness runs AWS's OpenTelemetry (ADOT) distro, so tracing to Datadog needs
        # environment variables only — no Datadog Agent, no Collector, no custom container.
        #
        #   * Protocol MUST be http/protobuf; the Datadog OTLP intake rejects gRPC.
        #   * service / env come from OTEL_SERVICE_NAME and OTEL_RESOURCE_ATTRIBUTES, not
        #     ddtrace's DD_SERVICE / DD_ENV — the OTel exporter ignores those.
        #   * compute_stats=true    -> Datadog computes APM trace metrics for these spans.
        #   * dd-otlp-source=llmobs -> the same spans also populate Agent Observability.
        #   * environmentVariables is a top-level harness param; environmentArtifact is only
        #     for a custom container image.
        #
        # The OTLP traces intake is in Preview and access-gated: the org must be allowlisted
        # or the intake returns 403 (contact your Datadog CSM).
        environmentVariables={
            "OTEL_EXPORTER_OTLP_TRACES_PROTOCOL": "http/protobuf",   # Datadog OTLP intake rejects gRPC
            "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT": f"https://trace.agent.{DD_SITE}/v1/traces",
            "OTEL_EXPORTER_OTLP_TRACES_HEADERS": f"dd-api-key={DD_API_KEY},compute_stats=true,dd-otlp-source=llmobs",
            "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",  # GenAI semconv 1.37+
            "OTEL_SERVICE_NAME": "anycompany-orchestrator",          # -> Datadog `service` + `ml_app`
            "OTEL_RESOURCE_ATTRIBUTES": "deployment.environment=demo,deployment.environment.name=demo",  # -> Datadog `env`
        },

    )
    print(f"  Harness updated with new system prompt.")
    print(f"  ID:  {HARNESS_ID}")
    print(f"  ARN: {HARNESS_ARN}")


Creating AgentCore Harness...

Harness created!
  Name:   anycompany_orchestrator
  ID:     anycompany_orchestrator-xx9W3rBi9G
  ARN:    arn:aws:bedrock-agentcore:us-west-2:359894365202:harness/anycompany_orchestrator-xx9W3rBi9G
  Status: CREATING


Poll until the Harness reaches READY status.

In [11]:
print("Waiting for harness to be ready...")
for _ in range(30):
    harness_info = agentcore_control.get_harness(harnessId=HARNESS_ID)
    status = harness_info["harness"]["status"]
    if status == "READY":
        print(f"Harness is READY!")
        break
    elif "FAIL" in status:
        reason = harness_info["harness"].get("failureReason", "unknown")
        raise RuntimeError(f"Harness creation failed: {status} — {reason}")
    print(f"  Status: {status} — waiting 10s...")
    time.sleep(10)
else:
    raise TimeoutError("Harness did not become ready within 5 minutes.")

print(f"\nHarness ID:  {HARNESS_ID}")
print(f"Harness ARN: {HARNESS_ARN}")

Waiting for harness to be ready...
  Status: CREATING — waiting 10s...
  Status: CREATING — waiting 10s...
Harness is READY!

Harness ID:  anycompany_orchestrator-xx9W3rBi9G
Harness ARN: arn:aws:bedrock-agentcore:us-west-2:359894365202:harness/anycompany_orchestrator-xx9W3rBi9G


## Step 6: Test the Harness

Invoke the Harness with a sample refund query and stream the response to confirm end-to-end orchestration.

In [12]:
session_id = str(uuid.uuid4()).upper()
# actorId scopes long-term memory (preferences, facts) per user.
# In the chatbot UI this is the Cognito custom:customer_id claim;
# in this notebook test we hardcode a Gold-tier user.
actor_id = "CUST-789"
print(f"Session ID: {session_id}")
print(f"Actor ID:   {actor_id}\n")

# Prepend the authenticated identity so the orchestrator knows who it is talking to.
# This mirrors the L5 chatbot (stream_orchestrator_to_ui), which injects the Cognito
# custom:customer_id claim. actorId alone only scopes long-term memory namespaces —
# the customer_id must be present IN the message for the orchestrator to scope order
# lookups (order lookups require customer_id). Without it the orchestrator has no
# authenticated customer_id to pass and will fail the lookup.
user_message = "Hi! I want to know my refund for ORD-10001"
augmented_message = f"[Authenticated customer: {actor_id}]\n\n{user_message}"

response = agentcore_runtime.invoke_harness(
    harnessArn=HARNESS_ARN,
    runtimeSessionId=session_id,
    actorId=actor_id,
    messages=[{"role": "user", "content": [{"text": augmented_message}]}],
)

# Process the streaming response
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

Session ID: F9A7F8CF-F188-40B7-9DAE-1FDC61AA96C9
Actor ID:   CUST-789

I'll help you check your refund eligibility for ORD-10001! Let me start by searching for the relevant specialist agents and getting today's date simultaneously.
[Tool: registry-tools___search_agents]

[Tool: registry-tools___search_agents]

[Tool: code_interpreter]


Today's date is **2026-10-07**. Now let me fetch the order details and refund policy simultaneously.
[Tool: registry-tools___invoke_agent]

[Tool: registry-tools___invoke_agent]


The Refund Agent returned general policy tiers but needs the customer's specific membership tier. Let me request that directly.
[Tool: registry-tools___invoke_agent]


The Refund Agent doesn't have a membership lookup tool, but it provided all tier-based rules. Let me check the Order Agent for the customer's membership tier.
[Tool: registry-tools___invoke_agent]


The Refund Agent provided all tier-based rules with their respective deadlines. Let me now calculate eligibility f

## Step 7: Save Configuration to SSM

Publish Harness and Memory IDs to SSM for use by L6 observability notebooks.

In [13]:
parameters = {
    f"{SSM_PREFIX}/harness_id": HARNESS_ID,
    f"{SSM_PREFIX}/harness_arn": HARNESS_ARN,
    f"{SSM_PREFIX}/harness_name": HARNESS_NAME,
    f"{SSM_PREFIX}/harness_role_arn": EXECUTION_ROLE_ARN,
    f"{SSM_PREFIX}/memory_id": MEMORY_ID,
    f"{SSM_PREFIX}/memory_arn": MEMORY_ARN,
    f"{SSM_PREFIX}/memory_name": MEMORY_NAME,
}

print(f"Saving {len(parameters)} parameters to SSM under '{SSM_PREFIX}/'\n")

for name, value in parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="String",
        Overwrite=True,
    )
    print(f"  ✓ {name} = {value}")

print(f"\nAll parameters saved.")

Saving 7 parameters to SSM under '/anycompany/agentcore/'

  ✓ /anycompany/agentcore/harness_id = anycompany_orchestrator-xx9W3rBi9G
  ✓ /anycompany/agentcore/harness_arn = arn:aws:bedrock-agentcore:us-west-2:359894365202:harness/anycompany_orchestrator-xx9W3rBi9G
  ✓ /anycompany/agentcore/harness_name = anycompany_orchestrator
  ✓ /anycompany/agentcore/harness_role_arn = arn:aws:iam::359894365202:role/AnyCompanyAgentCoreHarnessRole
  ✓ /anycompany/agentcore/memory_id = anycompany_agent_memory-qDC7LT5khH
  ✓ /anycompany/agentcore/memory_arn = arn:aws:bedrock-agentcore:us-west-2:359894365202:memory/anycompany_agent_memory-qDC7LT5khH
  ✓ /anycompany/agentcore/memory_name = anycompany_agent_memory

All parameters saved.


## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| IAM Role | `AnyCompanyAgentCoreHarnessRole` | Execution role for Bedrock, Memory, Registry, Gateway, and CloudWatch |
| AgentCore Memory | `anycompany_agent_memory` | Short-term chat history + long-term summarization, preferences, and semantic facts |
| AgentCore Harness | `anycompany_orchestrator` | Orchestrator running Claude Sonnet 4.6 — discovers and delegates to specialist agents |

---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- AgentCore: harness, memory
- IAM: harness execution role
- Observability: CloudWatch log delivery components and log groups (memory + gateway wiring)
- Configuration: SSM parameters

> Run this cleanup **before** the L3 #1 (`1_setup_resources.ipynb`) cleanup, since this notebook owns the gateway log delivery wiring (the gateway resource itself is owned by L3 #1).

Uncomment and run to delete all resources created in this notebook.

In [14]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the L3 #1 (1_setup_resources.ipynb) cleanup, since this
##       notebook owns the gateway log delivery wiring (the gateway itself is
##       owned by L3 #1).

# import boto3, os, time
#
## Note: In production, implement content filtering if responses may contain sensitive user data from memory or tool results.
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# HARNESS_NAME = "anycompany_orchestrator"
# MEMORY_NAME  = "anycompany_agent_memory"
# ROLE_NAME    = "AnyCompanyAgentCoreHarnessRole"
#
# iam               = boto3.client("iam")
# ssm               = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # Resolve IDs/ARNs from SSM, falling back to lookup-by-name (idempotent).
# HARNESS_ID  = _ssm_get("harness_id")
# HARNESS_ARN = _ssm_get("harness_arn")
# MEMORY_ID   = _ssm_get("memory_id")
# MEMORY_ARN  = _ssm_get("memory_arn")
# GATEWAY_ARN = _ssm_get("gateway_arn")
# GATEWAY_ID  = _ssm_get("gateway_id")
#
# if not (HARNESS_ID and HARNESS_ARN):
#     try:
#         for h in agentcore_control.list_harnesses().get("harnesses", []):
#             if h.get("harnessName") == HARNESS_NAME:
#                 HARNESS_ID  = h["harnessId"]
#                 HARNESS_ARN = h["arn"]
#                 break
#     except Exception as e:
#         print(f"List harnesses: {e}")
#
# if not (MEMORY_ID and MEMORY_ARN):
#     try:
#         for mem in agentcore_control.list_memories().get("memories", []):
#             try:
#                 detail = agentcore_control.get_memory(memoryId=mem["id"])
#                 if detail["memory"].get("name") == MEMORY_NAME:
#                     MEMORY_ID  = detail["memory"]["id"]
#                     MEMORY_ARN = detail["memory"]["arn"]
#                     break
#             except Exception:
#                 continue
#     except Exception as e:
#         print(f"List memories: {e}")
#
# # ── Phase 1: Delete Harness (must precede Memory + Role delete) ─────────────
# if HARNESS_ID:
#     try:
#         agentcore_control.delete_harness(harnessId=HARNESS_ID)
#         print(f"Initiated delete of Harness: {HARNESS_ID}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_harness(harnessId=HARNESS_ID)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Harness fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for Harness to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Harness {HARNESS_ID} already deleted.")
#     except Exception as e:
#         print(f"Harness delete: {e}")
# else:
#     print(f"Harness {HARNESS_NAME} not found (already deleted).")
#
# # Helper: drain log delivery components for a given resource ARN.
# # IMPORTANT: filter destinations to ours-only (name pattern + skip XRAY type)
# # to avoid deleting the shared XRAY destination used by other agents.
# def _drain_log_delivery(resource_arn, dest_name_prefix):
#     if not resource_arn:
#         return
#     # 1. Find sources tied to this resource
#     sources_for_res = []
#     try:
#         paginator = logs_client.get_paginator("describe_delivery_sources")
#         for page in paginator.paginate():
#             for s in page.get("deliverySources", []):
#                 if resource_arn in s.get("resourceArns", []):
#                     sources_for_res.append(s)
#     except Exception as e:
#         print(f"List delivery sources for {resource_arn}: {e}")
#         return
#     source_names = {s["name"] for s in sources_for_res}
#     # 2. Delete deliveries originating from those sources, collect dest ARNs
#     dest_arns_seen = set()
#     if source_names:
#         try:
#             paginator = logs_client.get_paginator("describe_deliveries")
#             for page in paginator.paginate():
#                 for d in page.get("deliveries", []):
#                     if d.get("deliverySourceName") in source_names:
#                         if d.get("deliveryDestinationArn"):
#                             dest_arns_seen.add(d["deliveryDestinationArn"])
#                         try:
#                             logs_client.delete_delivery(id=d["id"])
#                             print(f"Deleted delivery: {d['id']}")
#                         except Exception as e:
#                             print(f"Delete delivery ({d['id']}): {e}")
#         except Exception as e:
#             print(f"List deliveries: {e}")
#     # 3. Delete destinations — but ONLY ones we own (matched by name prefix
#     #    and not XRAY type) so we never touch the shared XRAY destination.
#     try:
#         all_dests = logs_client.describe_delivery_destinations().get("deliveryDestinations", [])
#         dests_by_arn = {d["arn"]: d for d in all_dests}
#     except Exception as e:
#         print(f"List delivery destinations: {e}")
#         dests_by_arn = {}
#     for dest_arn in dest_arns_seen:
#         d = dests_by_arn.get(dest_arn)
#         if not d:
#             continue
#         if d.get("deliveryDestinationType") == "XRAY":
#             continue  # shared, never delete
#         dname = d.get("name", "")
#         if not dname.startswith(dest_name_prefix):
#             continue  # not ours
#         try:
#             logs_client.delete_delivery_destination(name=dname)
#             print(f"Deleted delivery destination: {dname}")
#         except Exception as e:
#             print(f"Delete delivery destination ({dname}): {e}")
#     # 4. Delete sources
#     for s in sources_for_res:
#         try:
#             logs_client.delete_delivery_source(name=s["name"])
#             print(f"Deleted delivery source: {s['name']}")
#         except Exception as e:
#             print(f"Delete delivery source ({s['name']}): {e}")
#
# # ── Phase 2: Drain Memory CloudWatch log delivery ───────────────────────────
# if MEMORY_ARN:
#     _drain_log_delivery(MEMORY_ARN, dest_name_prefix="mem-")
# else:
#     print("Memory ARN not resolved; skipping memory log delivery cleanup.")
#
# # ── Phase 3: Drain Gateway CloudWatch log delivery (wiring only) ────────────
# if GATEWAY_ARN:
#     _drain_log_delivery(GATEWAY_ARN, dest_name_prefix="gw-")
# else:
#     print("Gateway ARN not resolved; skipping gateway log delivery cleanup.")
#
# # ── Phase 4: Delete Memory and poll until gone ─────────────────────────────
# if MEMORY_ID:
#     try:
#         agentcore_control.delete_memory(memoryId=MEMORY_ID)
#         print(f"Initiated delete of Memory: {MEMORY_ID}")
#         deadline = time.time() + 300
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_memory(memoryId=MEMORY_ID)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Memory fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for Memory to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Memory {MEMORY_ID} already deleted.")
#     except Exception as e:
#         print(f"Memory delete: {e}")
# else:
#     print(f"Memory {MEMORY_NAME} not found (already deleted).")
#
# # ── Phase 5: Delete CloudWatch log groups ──────────────────────────────────
# log_groups = []
# if MEMORY_ID:
#     log_groups.append(f"/aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/{MEMORY_ID}")
# if GATEWAY_ID:
#     log_groups.append(f"/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{GATEWAY_ID}")
# for lg in log_groups:
#     try:
#         logs_client.delete_log_group(logGroupName=lg)
#         print(f"Deleted log group: {lg}")
#     except logs_client.exceptions.ResourceNotFoundException:
#         pass
#     except Exception as e:
#         print(f"Log group delete ({lg}): {e}")
#
# # ── Phase 6: Delete IAM Role (used by both harness and memory) ─────────────
# try:
#     for ap in iam.list_attached_role_policies(RoleName=ROLE_NAME).get("AttachedPolicies", []):
#         iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=ap["PolicyArn"])
#     for p in iam.list_role_policies(RoleName=ROLE_NAME).get("PolicyNames", []):
#         iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=p)
#     iam.delete_role(RoleName=ROLE_NAME)
#     print(f"Deleted IAM role: {ROLE_NAME}")
# except iam.exceptions.NoSuchEntityException:
#     print(f"IAM role {ROLE_NAME} already deleted.")
# except Exception as e:
#     print(f"IAM role cleanup: {e}")
#
# # ── Phase 7: Delete SSM parameters ─────────────────────────────────────────
# for param in [
#     "harness_id", "harness_arn", "harness_name", "harness_role_arn",
#     "memory_id", "memory_arn", "memory_name",
# ]:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"Deleted SSM parameter: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#     except Exception as e:
#         print(f"SSM delete ({param}): {e}")
#
# print("\n✅ All Orchestrator resources cleaned up.")